# RGBD perception

This notebook visualizes the RGBD feed, segments a coffee mug candidate on the table, and can also segment the robot end-effector or a colored marker on it.


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Import display, image, and robot helpers.


In [ ]:
import base64
import json
import time

import cv2
import ipywidgets as widgets
import numpy as np
from IPython.display import display

from sdk_client import Robot


Create the robot RGBD client. Override `G1_RGBD_HOST` and `G1_RGBD_PORT` if the image server runs elsewhere.


In [ ]:
RGBD_HOST = os.environ.get("G1_RGBD_HOST", "192.168.2.41")
RGBD_PORT = int(os.environ.get("G1_RGBD_PORT", "5555"))
RGBD_TOPIC = os.environ.get("G1_RGBD_TOPIC", "")
robot = Robot(
    iface=IFACE,
    domain_id=DOMAIN_ID,
    safety_boot=False,
    recover_dev_mode_on_init=False,
    auto_start_sensors=False,
    rgbd_host=RGBD_HOST,
    rgbd_port=RGBD_PORT,
    rgbd_topic=RGBD_TOPIC,
)
print(f"RGBD client ready for tcp://{RGBD_HOST}:{RGBD_PORT} topic={RGBD_TOPIC!r}")


Helper functions for HTML image display and a simple visibility detector.


In [ ]:
def jpeg_data_url(bgr):
    # TODO: JPEG-encode the OpenCV BGR image and wrap it as a browser data URL.
    raise NotImplementedError("Participant exercise: complete jpeg_data_url.")


def colorize_depth(depth_m, max_depth_m=4.0):
    # TODO: Convert valid depth values into an 8-bit display image and apply a colormap.
    raise NotImplementedError("Participant exercise: complete colorize_depth.")


def detect_visible_object(rgb_bgr, depth_m, hsv_low, hsv_high, min_area_px=500, max_depth_m=2.0):
    # TODO: Inspect RGB-D data and return a compact visible-object summary.
    raise NotImplementedError("Participant exercise: complete detect_visible_object.")


Run the viewer. The default HSV range detects many red objects; tune the sliders for the object used in your exercise.


In [ ]:
h_low = widgets.IntSlider(value=0, min=0, max=179, description="H low")
h_high = widgets.IntSlider(value=179, min=0, max=179, description="H high")
s_low = widgets.IntSlider(value=0, min=0, max=255, description="S low")
s_high = widgets.IntSlider(value=255, min=0, max=255, description="S high")
v_low = widgets.IntSlider(value=80, min=0, max=255, description="V low")
v_high = widgets.IntSlider(value=255, min=0, max=255, description="V high")
min_area = widgets.IntSlider(value=500, min=50, max=10000, step=50, description="Area")
max_depth = widgets.FloatSlider(value=2.0, min=0.2, max=6.0, step=0.1, description="Depth m")
refresh = widgets.Button(description="Refresh Frame", button_style="success")
rgb_img = widgets.HTML(value="")
depth_img = widgets.HTML(value="")
status = widgets.HTML(value="")


def update(_=None):
    # TODO: Capture an RGB-D frame, run the visible-object detector, and update the image/status widgets.
    raise NotImplementedError("Participant exercise: complete update.")

refresh.on_click(update)
update()
display(widgets.VBox([
    widgets.HBox([h_low, h_high, s_low, s_high, v_low, v_high]),
    widgets.HBox([min_area, max_depth, refresh]),
    status,
    widgets.HBox([rgb_img, depth_img]),
]))


## Image segmentation and end-effector detection

This cell segments the RGBD frame by HSV and depth. Use the object controls for the table object and the end-effector controls for the robot hand or a colored marker on it.


In [ ]:
def segment_hsv_depth(rgb_bgr, depth_m, hsv_low, hsv_high, *, min_area_px=300, max_depth_m=2.5, roi=None, label="segment", color=(0, 255, 255)):
    # TODO: Threshold HSV color, combine it with valid depth, clean the mask, and extract contour detections.
    raise NotImplementedError("Participant exercise: complete segment_hsv_depth.")


def mask_data_url(mask):
    # TODO: Convert a grayscale mask into a browser-displayable JPEG data URL.
    raise NotImplementedError("Participant exercise: complete mask_data_url.")

obj_h_low = widgets.IntSlider(value=0, min=0, max=179, description="Obj H low")
obj_h_high = widgets.IntSlider(value=179, min=0, max=179, description="Obj H high")
obj_s_low = widgets.IntSlider(value=0, min=0, max=255, description="Obj S low")
obj_v_low = widgets.IntSlider(value=80, min=0, max=255, description="Obj V low")
ee_h_low = widgets.IntSlider(value=0, min=0, max=179, description="EE H low")
ee_h_high = widgets.IntSlider(value=179, min=0, max=179, description="EE H high")
ee_s_low = widgets.IntSlider(value=0, min=0, max=255, description="EE S low")
ee_s_high = widgets.IntSlider(value=255, min=0, max=255, description="EE S high")
ee_v_low = widgets.IntSlider(value=0, min=0, max=255, description="EE V low")
ee_v_high = widgets.IntSlider(value=90, min=0, max=255, description="EE V high")
seg_min_area = widgets.IntSlider(value=300, min=50, max=8000, step=50, description="Area")
seg_depth = widgets.FloatSlider(value=2.5, min=0.2, max=6.0, step=0.1, description="Depth m")
seg_refresh = widgets.Button(description="Segment Frame", button_style="success")
seg_overlay_img = widgets.HTML(value="")
obj_mask_img = widgets.HTML(value="")
ee_mask_img = widgets.HTML(value="")
seg_status = widgets.Textarea(layout=widgets.Layout(width="100%", height="180px"), disabled=True)


def update_segmentation(_=None):
    # TODO: Capture RGB-D, run both segmenters, update images, and print detection JSON.
    raise NotImplementedError("Participant exercise: complete update_segmentation.")

seg_refresh.on_click(update_segmentation)
display(widgets.VBox([
    widgets.HBox([obj_h_low, obj_h_high, obj_s_low, obj_v_low]),
    widgets.HBox([ee_h_low, ee_h_high, ee_s_low, ee_s_high, ee_v_low, ee_v_high]),
    widgets.HBox([seg_min_area, seg_depth, seg_refresh]),
    seg_status,
    widgets.HBox([seg_overlay_img, obj_mask_img, ee_mask_img]),
]))
